# `06-deepagent/test.ipynb`
1. csv 파일을 받아서
2. 데이터 분석을 진행하고
3. 슬랙으로 결과를 보냄

```sh
uv pip install deepagents slack-sdk langcahain-daytona langchain-openai
```

In [30]:
from dotenv import load_dotenv

load_dotenv()

True

### Daytona Sandbox

In [20]:
from daytona import Daytona
from langchain_daytona import DaytonaSandbox

sandbox = Daytona().create()
backend = DaytonaSandbox(sandbox=sandbox)

In [21]:
result = backend.execute('ls -a')
print(result.output)

.
..
.bash_logout
.bashrc
.daytona
.face
.face.icon
.profile
.zshrc


In [ ]:
import csv
import io

data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

buf = io.StringIO()
writer = csv.writer(buf)
writer.writerows(data)

# 원본 csv 데이터
csv_bytes = buf.getvalue().encode('utf-8')
buf.close()

# 파일 저장(지금 우리는 필요 없음)
with open('./sales.csv', 'wb') as f:
    f.write(csv_bytes)

# sandbox 에 업로드
backend.upload_files(
    [
        ('/home/daytona/data/sales_data.csv', csv_bytes)
    ]
)


[FileUploadResponse(path='/home/daytona/data/sales_data.csv', error=None)]

### Slack
- Agent 가 사용할 Slack Messaging Tool 생성

In [ ]:
import os
from slack_sdk import WebClient

SLACK_BOT_TOKEN = os.getenv('SLACK_BOT_TOKEN')

client = WebClient(token=SLACK_BOT_TOKEN)

# 봇이 접근 가능한 모든 채널 리스트
res = client.conversations_list()
for ch in res['channels']:
    print(ch['name'], ch['id'])

social_channel_id = 'C0A3WG6GLQK'

client.chat_postMessage(
    channel=social_channel_id,
    text='hello'
)

소셜 C0A3WG6GLQK
새-채널 C0A40SZEUB0
새-워크스페이스-전체 C0A4X78KVNU


In [41]:
import os
from slack_sdk import WebClient
from langchain.tools import tool

SLACK_TOKEN = os.getenv('SLACK_BOT_TOKEN')
slack_client = WebClient(token=SLACK_TOKEN)


@tool(parse_docstring=True)  # LLM에게 Tool 설명을 docstring 에서 제공
def send_slack_message(text: str, file_path: str | None = None) -> str:
    """메세지 전송, 경우에 따라 이미지같은 파일을 첨부함
    
    Args:
        text: (str) 메세지의 내용
        file_path: (str) 파일시스템 상 첨부파일의 경로
    """

    social_channel_id = 'C0A3WG6GLQK'
    # 첨부 파일 없으면
    if not file_path:
        slack_client.chat_postMessage(channel=social_channel_id, text=text)
    else:
        fp = backend.download_files([file_path])
        slack_client.files_upload_v2(
            channel=social_channel_id,
            content=fp[0].content,
            initial_comment=text,
        )
    
    return 'Message sent.'

## Build Deep Agent

In [44]:
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model='openai:gpt-5-mini',
    tools=[send_slack_message],
    backend=backend,
    checkpointer=checkpointer,
)

thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

In [46]:
input_message = {
    'role': 'user',
    'content': '''
    현재 폴더 안에 ./data/sales_data.csv 파일을 분석하고, 시각화 해줘.
    다 끝나면 분석결과와 시각화 이미지를 Slack 메세지로 보내줘.
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0a1c7b8fac0efc4f0069afa76e9d508190b48926c45ce241e9', 'summary': [], 'type': 'reasoning'}, {'arguments': '{"file_path":"/data/sales_data.csv","offset":0,"limit":200}', 'call_id': 'call_bQgbg7dSZC1AGnhE9FFN5sDP', 'name': 'read_file', 'type': 'function_call', 'id': 'fc_0a1c7b8fac0efc4f0069afa771263081908c7b141a3685f84b', 'status': 'completed'}]
Tool Calls:
  read_file (call_bQgbg7dSZC1AGnhE9FFN5sDP)
 Call ID: call_bQgbg7dSZC1AGnhE9FFN5sDP
  Args:
    file_path: /data/sales_data.csv
    offset: 0
    limit: 200
================================= Tool Message =================================
Name: read_file

Error: File '/data/sales_data.csv' not found
================================== Ai Message ==================================

[{'id': 'rs_0a1c7b8fac0efc4f0069afa7740c708190a7550bd51b6fa4f7', 'summary': [], 'type': 'reasoning'}, {'arguments': '{"path":"/"}', 'call_id': 'call_GRoh4BZNax1Cg4Olxpk

In [49]:
input_message = {
    'role': 'user',
    'content': '''
    채널은 기본 세팅 되어있음. 2번 데이터 시각화 해서 슬랙 메시지로 보내줘
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0a1c7b8fac0efc4f0069afaa0f27408190866933ede584285f', 'summary': [], 'type': 'reasoning'}, {'arguments': '{"text":"분석 결과 요약:\\n\\n총 매출(Revenue): 840.00\\n총 판매 수량(Units Sold): 33\\n\\n제품별 매출:\\nWidget A: 425\\nWidget B: 325\\nWidget C: 90\\n\\n날짜별 판매 수량:\\n2025-08-01: 10\\n2025-08-02: 5\\n2025-08-03: 7\\n2025-08-04: 3\\n2025-08-05: 8","file_path":"/home/daytona/data/sales_analysis.png"}', 'call_id': 'call_zfiQgNcAdjePLWSyYfNb28ta', 'name': 'send_slack_message', 'type': 'function_call', 'id': 'fc_0a1c7b8fac0efc4f0069afaa1193f08190859cc32797a01ab9', 'status': 'completed'}]
Tool Calls:
  send_slack_message (call_zfiQgNcAdjePLWSyYfNb28ta)
 Call ID: call_zfiQgNcAdjePLWSyYfNb28ta
  Args:
    text: 분석 결과 요약:

총 매출(Revenue): 840.00
총 판매 수량(Units Sold): 33

제품별 매출:
Widget A: 425
Widget B: 325
Widget C: 90

날짜별 판매 수량:
2025-08-01: 10
2025-08-02: 5
2025-08-03: 7
2025-08-04: 3
2025-08-05: 8
    file_path: /h